# RuBioRoBERTa в Google Colab

1. Трансформер **без файнтюна** → MLP поверх замороженных эмбеддингов.
2. **Файнтюн** RuBioRoBERTa (LoRA + взвешенный loss), затем Optuna.

**Перед запуском:** Runtime → Change runtime type → GPU.

In [ ]:
!nvidia-smi

In [ ]:
!pip install -q transformers accelerate peft optuna

In [ ]:
import sys

!git clone -q https://github.com/vinograd131/anamnesis.git
%cd anamnesis
if "/content/anamnesis" not in sys.path:
    sys.path.insert(0, "/content/anamnesis")

## 1. Без файнтюна: frozen-эмбеддинги + MLP

Эмбеддинги кэшируются в `models/` — повторный запуск в этой сессии мгновенный.

In [ ]:
from src.transformer_frozen import main as frozen_main
frozen_main("dev")

## 2. Файнтюн (LoRA + взвешенный loss)

Один прогон с дефолтными гиперпараметрами — проверяем, что loss падает и macro-F1 считается.

In [ ]:
from src.transformer_ft import train_once
train_once(eval_split="dev", use_lora=True, save=True)

### Optuna (HPO)

После успешного одиночного прогона. Долго — начни с небольшого числа трайлов.

In [ ]:
from src.transformer_ft import run_optuna
study = run_optuna(n_trials=20, eval_split="dev", use_lora=True)

## Метрики

Файлы в Colab временные — скачай `reports/metrics.json`, чтобы забрать результаты.

In [ ]:
import json
print(json.dumps(json.load(open("reports/metrics.json")), ensure_ascii=False, indent=2))

In [ ]:
from google.colab import files
files.download("reports/metrics.json")